# 06 - Online recalibration

A plant does not stand still: substrate quality changes with the harvest, the
biology adapts, a digester is emptied. A model calibrated in spring drifts away
from the plant by autumn.

`OnlineCalibrator` is built for that case. It differs from the initial
calibration in two ways:

- it starts from the **currently used** parameters instead of a default,
- it **refuses** to move them far, or at all, when the evidence is thin.

Those guardrails are the point: an automatic recalibration that follows every
sensor glitch is worse than none.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from demo_plant import build_demo_plant, make_twin_measurements, simulate
from matplotlib.ticker import MaxNLocator

from pyadm1ode_calibration import OnlineCalibrator

# A plant whose hydrolysis slowly gets worse: 2.0 -> 3.0 over three periods.
DRIFT = [2.0, 2.5, 3.0]
windows = [
    make_twin_measurements(days=3, noise=0.02, seed=10 + i, parameters={"k_hyd_ch": value})
    for i, value in enumerate(DRIFT)
]
print("true k_hyd_ch per window:", DRIFT)

## Following the drift

We start from the value the last initial calibration produced and hand the
calibrator one window after another. Exactly how it would run in operation,
once a week or once a month.

In [ ]:
current = {"k_hyd_ch": 2.0}
trajectory = [current["k_hyd_ch"]]

for i, window in enumerate(windows, start=1):
    result = OnlineCalibrator(build_demo_plant(days=3), verbose=False).calibrate(
        window,
        parameters=["k_hyd_ch"],
        current_parameters=current,
        objectives=["Q_gas"],
        max_parameter_change=0.20,
        variance_threshold=0.15,
        max_iterations=15,
        use_constraints=False,
    )
    current = dict(result.parameters)
    trajectory.append(current["k_hyd_ch"])
    print(f"window {i}: true {DRIFT[i - 1]:.2f} -> tracked {current['k_hyd_ch']:.3f}   ({result.message})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.step(range(len(trajectory)), trajectory, where="post", marker="o", label="tracked")
ax.step(range(1, len(DRIFT) + 1), DRIFT, where="post", ls="--", color="tab:green", label="true")
ax.set_xlabel("recalibration round")
# Rounds are counts, so half-integer ticks would be meaningless.
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylabel("k_hyd_ch")
ax.set_title("Parameter tracking a drifting plant")
ax.legend()
fig.tight_layout()

## What the tracking does to the model output

The parameter plot above shows the calibrator following the drift, but not
whether the *model* got better. The panels below answer that directly. For every
window the measured `Q_gas` against the simulation with the parameters the round
**started from** and the ones it **returned**.


In [ ]:
# Two simulations per window, so 6 short ones in total. trajectory[i-1] is what round
# i started from and trajectory[i] what it returned, so nothing has to be carried out
# of the loop above.
CHANNEL = "Q_gas"


def rmse(y, reference):
    return float(np.sqrt(np.mean((y - reference) ** 2)))

plant = build_demo_plant(days=3)

fig, axes = plt.subplots(1, len(windows), figsize=(11, 3.2), sharey=True)
for i, (ax, window) in enumerate(zip(np.atleast_1d(axes), windows), start=1):
    before, after = trajectory[i - 1], trajectory[i]
    t = window.data.index
    measured = window.data[CHANNEL].to_numpy()
    sim_before = np.asarray(
        simulate(plant, window, {"k_hyd_ch": before})[CHANNEL], dtype=float
    )
    sim_after = np.asarray(
        simulate(plant, window, {"k_hyd_ch": after})[CHANNEL], dtype=float
    )

    ax.plot(t, measured, color="0.45", lw=0.9, alpha=0.8, label="measured")
    ax.plot(t, sim_before, ls="--", color="tab:red", label=f"before ({before:.2f})")
    ax.plot(t, sim_after, color="tab:blue", label=f"after ({after:.2f})")
    plateau = measured[12:]
    ax.set_xlim(t[0], t[-1])   # no empty quarter panel
    ax.set_ylim(plateau.min() * 0.97, plateau.max() * 1.06)
    ax.set_title(f"round {i}  ·  true {DRIFT[i - 1]:.2f}", fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.legend(fontsize=7, loc="lower right")

    print(
        f"round {i}: RMSE(Q_gas) {rmse(sim_before, measured):7.1f}"
        f" -> {rmse(sim_after, measured):7.1f} m3/d"
    )

np.atleast_1d(axes)[0].set_ylabel(f"{CHANNEL} [m3/d]")
fig.suptitle("Measurement against simulation, before and after each recalibration")
fig.tight_layout()
